# Marketing Silver Transformation
Run in Databricks No deduplication or imputation. Raw fields remain authoritative for investigation.

Date format defaults to AUTO: recognizes ISO year-month-day and day-month-year (dash or slash). Day-first interpretation is explicit; use MM/dd/yyyy or MM-dd-yyyy if your source is month-first. Parsing does not establish whether these are genuine campaign dates. Acquisition cost and ROI definitions still require source verification.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import re

dbutils.widgets.text("catalog", "marketing_catalog")
dbutils.widgets.text("date_format", "AUTO")
catalog = dbutils.widgets.get("catalog").strip()
date_format = dbutils.widgets.get("date_format").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name")
allowed_formats = {"AUTO", "yyyy-MM-dd", "dd-MM-yyyy", "dd/MM/yyyy", "MM/dd/yyyy", "MM-dd-yyyy"}
if date_format not in allowed_formats:
    raise ValueError(f"Choose date_format from {sorted(allowed_formats)}")
source_table = f"{catalog}.marketing_bronze.campaigns"
target_table = f"{catalog}.marketing_silver.campaigns"
print("Source:", source_table)
print("Target:", target_table)
print("Date format:", date_format)

Source: marketing_catalog.marketing_bronze.campaigns
Target: marketing_catalog.marketing_silver.campaigns
Date format: AUTO


In [0]:
bronze_df = spark.table(source_table)
expected = {"Campaign_ID", "Company", "Campaign_Type", "Target_Audience", "Duration", "Channel_Used", "Conversion_Rate", "Acquisition_Cost", "ROI", "Location", "Language", "Clicks", "Impressions", "Engagement_Score", "Customer_Segment", "Date", "_source_file", "_ingest_batch_id", "_ingested_at_utc"}
if set(bronze_df.columns) != expected:
    raise ValueError(f"Unexpected Bronze schema. Missing={sorted(expected-set(bronze_df.columns))}; Extra={sorted(set(bronze_df.columns)-expected)}")
bronze_count = bronze_df.count()
if bronze_count == 0:
    raise ValueError("Bronze is empty; stopping before overwrite")
display(bronze_df.select("Date", "Duration", "Acquisition_Cost").limit(20))
print("Bronze rows:", bronze_count)

Date,Duration,Acquisition_Cost
2021-01-01,30 days,"$16,174.00"
2021-01-02,60 days,"$11,566.00"
2021-01-03,30 days,"$10,200.00"
2021-01-04,60 days,"$12,724.00"
2021-01-05,15 days,"$16,452.00"
2021-01-06,15 days,"$9,716.00"
2021-01-07,60 days,"$11,067.00"
2021-01-08,45 days,"$13,280.00"
2021-01-09,15 days,"$18,066.00"
2021-01-10,15 days,"$13,766.00"


Bronze rows: 200000


In [0]:
numeric_fields = {
    "Campaign_ID": ("campaign_id_typed", "BIGINT"),
    "Conversion_Rate": ("conversion_rate_typed", "DOUBLE"),
    "ROI": ("roi_typed", "DOUBLE"),
    "Clicks": ("clicks_typed", "BIGINT"),
    "Impressions": ("impressions_typed", "BIGINT"),
    "Engagement_Score": ("engagement_score_typed", "INT")
}
silver_df = bronze_df
for source, (target, dtype) in numeric_fields.items():
    silver_df = silver_df.withColumn(target, F.expr(f"try_cast(trim(`{source}`) AS {dtype})"))
date_text = F.trim(F.col("Date"))
if date_format == "AUTO":
    parsed_date = F.coalesce(*[
        F.try_to_timestamp(date_text, F.lit(fmt)).cast("date")
        for fmt in ["yyyy-MM-dd", "dd-MM-yyyy", "dd/MM/yyyy"]
    ])
else:
    parsed_date = F.try_to_timestamp(date_text, F.lit(date_format)).cast("date")
silver_df = (silver_df
    .withColumn("campaign_date", parsed_date)
    .withColumn("duration_days", F.when(
        F.trim(F.col("Duration")).rlike(r"(?i)^\d+\s+days?$"),
        F.regexp_extract(F.trim(F.col("Duration")), r"^(\d+)", 1)))
    .withColumn("duration_days", F.expr("try_cast(duration_days AS INT)"))
    .withColumn("_cost_text", F.regexp_replace(F.trim(F.col("Acquisition_Cost")), r"[$,]", ""))
    .withColumn("acquisition_cost_typed", F.expr("try_cast(_cost_text AS DECIMAL(18,2))"))
    .drop("_cost_text")
    .withColumn("_silver_processed_at_utc", F.lit(datetime.now(timezone.utc)).cast("timestamp"))
)

In [0]:
typed_fields = ["campaign_id_typed", "campaign_date", "duration_days", "conversion_rate_typed", "acquisition_cost_typed", "roi_typed", "clicks_typed", "impressions_typed", "engagement_score_typed"]
checks = [F.when(F.col(x).isNull(), F.lit(f"missing_or_invalid:{x}")) for x in typed_fields]
ranges = {
    "campaign_id_nonpositive": F.col("campaign_id_typed") <= 0,
    "duration_nonpositive": F.col("duration_days") <= 0,
    "conversion_rate_out_of_range": ~F.col("conversion_rate_typed").between(0, 1),
    "negative_acquisition_cost": F.col("acquisition_cost_typed") < 0,
    "negative_clicks": F.col("clicks_typed") < 0,
    "negative_impressions": F.col("impressions_typed") < 0,
    "clicks_exceed_impressions": F.col("clicks_typed") > F.col("impressions_typed"),
    "nonfinite_conversion_rate": F.isnan("conversion_rate_typed") | (F.abs(F.col("conversion_rate_typed")) == float("inf")),
    "nonfinite_roi": F.isnan("roi_typed") | (F.abs(F.col("roi_typed")) == float("inf"))
}
checks += [F.when(condition, F.lit(label)) for label, condition in ranges.items()]
silver_df = (silver_df
    .withColumn("_quality_issues", F.filter(F.array(*checks), lambda x: x.isNotNull()))
    .withColumn("_quality_passed", F.size("_quality_issues") == 0)
)
display(silver_df.groupBy("_quality_passed").count())
display(silver_df.select(F.explode("_quality_issues").alias("issue")).groupBy("issue").count())
# Diagnostic only: repeated IDs remain in Silver.
display(silver_df.groupBy("campaign_id_typed").count().filter(F.col("count") > 1).limit(20))

_quality_passed,count
true,200000


issue,count


campaign_id_typed,count


In [0]:
silver_count = silver_df.count()
if silver_count != bronze_count or silver_count == 0:
    raise ValueError(f"Row mismatch: Bronze={bronze_count}, Silver={silver_count}")
if len(silver_df.columns) != 31:
    raise ValueError(f"Expected 31 Silver columns; found {len(silver_df.columns)}")
if silver_df.filter(F.col("campaign_date").isNotNull()).limit(1).count() == 0:
    raise ValueError("No dates parsed. Correct the date_format widget and rerun all cells.")
display(silver_df.select(
    F.min("campaign_date").alias("earliest_date"),
    F.max("campaign_date").alias("latest_date"),
    F.sum(F.when(F.col("campaign_date").isNull(), 1).otherwise(0)).alias("unparsed_dates")
))
display(silver_df.select("Date", "campaign_date", "Duration", "duration_days", "Acquisition_Cost", "acquisition_cost_typed").limit(20))

earliest_date,latest_date,unparsed_dates
2021-01-01,2021-12-31,0


Date,campaign_date,Duration,duration_days,Acquisition_Cost,acquisition_cost_typed
2021-01-01,2021-01-01,30 days,30,"$16,174.00",16174.00
2021-01-02,2021-01-02,60 days,60,"$11,566.00",11566.00
2021-01-03,2021-01-03,30 days,30,"$10,200.00",10200.00
2021-01-04,2021-01-04,60 days,60,"$12,724.00",12724.00
2021-01-05,2021-01-05,15 days,15,"$16,452.00",16452.00
2021-01-06,2021-01-06,15 days,15,"$9,716.00",9716.00
2021-01-07,2021-01-07,60 days,60,"$11,067.00",11067.00
2021-01-08,2021-01-08,45 days,45,"$13,280.00",13280.00
2021-01-09,2021-01-09,15 days,15,"$18,066.00",18066.00
2021-01-10,2021-01-10,15 days,15,"$13,766.00",13766.00


## Save validated snapshot
This write replaces only the marketing Silver table, including its schema. Bronze remains unchanged. Quality issues are retained, not filtered. Review the date parsing and quality diagnostics before using the data for reporting or ML.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.marketing_silver")
(silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table))
saved_df = spark.table(target_table)
saved_count = saved_df.count()
if saved_count != bronze_count or len(saved_df.columns) != 31:
    raise RuntimeError("Saved Silver validation failed")
print("Validation: PASSED")
print("Saved table:", target_table)
print("Bronze rows:", bronze_count)
print("Silver rows:", saved_count)
print("Silver columns:", len(saved_df.columns))
display(saved_df.select(
    F.min("campaign_date").alias("earliest_date"),
    F.max("campaign_date").alias("latest_date"),
    F.sum(F.when(~F.col("_quality_passed"), 1).otherwise(0)).alias("flagged_rows")
))
display(saved_df.select("campaign_id_typed", "campaign_date", "duration_days", "acquisition_cost_typed", "conversion_rate_typed", "_quality_passed").limit(10))

Validation: PASSED
Saved table: marketing_catalog.marketing_silver.campaigns
Bronze rows: 200000
Silver rows: 200000
Silver columns: 31


earliest_date,latest_date,flagged_rows
2021-01-01,2021-12-31,0


campaign_id_typed,campaign_date,duration_days,acquisition_cost_typed,conversion_rate_typed,_quality_passed
1,2021-01-01,30,16174.00,0.04,true
2,2021-01-02,60,11566.00,0.12,true
3,2021-01-03,30,10200.00,0.07,true
4,2021-01-04,60,12724.00,0.11,true
5,2021-01-05,15,16452.00,0.05,true
6,2021-01-06,15,9716.00,0.07,true
7,2021-01-07,60,11067.00,0.13,true
8,2021-01-08,45,13280.00,0.08,true
9,2021-01-09,15,18066.00,0.09,true
10,2021-01-10,15,13766.00,0.09,true


## Interpretation and limitations
- 31 columns = 19 Bronze fields + 9 typed fields + Silver timestamp + 2 quality fields.
- Campaign means are not denominator-weighted conversion rates. Do not invent conversion counts.
- Acquisition cost is not assumed to be campaign spend; ROI is not used to invent revenue.
- Quality flags cover selected parse/range rules, not all possible business validation.
- This notebook is a batch snapshot pipeline, not incremental ingestion or production deployment.
- Date source provenance and duplicate-ID meaning still need review before time-based ML.


Original column	Added Silver column	Change
<div> Campaign_ID	--> ampaign_id_typed --> Converts to integer </div>
<div> Date --> campaign_date	--> Converts text to date</div>
<div> Duration --> duration_days	--> Converts "30 days" → 30</div>
<div> Acquisition_Cost --> acquisition_cost_typed	--> Converts "$16,174.00" → 16174.00</div>
<div> Conversion_Rate	--> conversion_rate_typed	--> Converts to decimal number</div>
<div> ROI	roi_typed	--> Converts to decimal number</div>
<div> Clicks	clicks_typed	--> Converts to integer</div>
<div> Impressions	impressions_typed	--> Converts to integer</div>
<div> Engagement_Score --> engagement_score_typed -->	Converts to integer</div>

Also adds:
- _silver_processed_at_utc: processing timestamp.
- _quality_issues: missing/invalid values and selected range violations.
- _quality_passed: whether the row passed those checks.